
# Lab 1: Creating Review Gold Layer

In this notebook, we will create monthly aggregration of TMSIS and store the table in the gold layer.
This gold table will be used later to power a dashboard.

## Objectives
2. **Data Transformation**: Aggregate the TMSIS data by month and adding the total amount of the claims.
3. **Create a New Table**: Create a new table in the gold layer to store the data.

Let's get started!

####Setup
First step in all the labs will be to run a setup notebook. This notebook to create a catalog to store data and to set your default catalog 

In [ ]:
%run "../Lab 0 - Setup/SetCatalogName"


For this gold table, we are going to use AI to summarize the review. Instead of creating a summary for each review and then a summary of the summary, we are going to create a temporary view that for each unique rating will have a single row of all the reviews.

A Spark temporary view is a virtual table created using a SQL query. It exists only for the duration of the Spark session and is not persisted.

In [ ]:
%sql
create or replace temp view collected_reviews
as
select
       rating,
       collect_set(review) as combined_ratings
from silver.medical_providers
group by rating

Using the temporary view we can use builtin Databricks AI functions.

Databricks AI functions allow users to leverage generative AI directly in SQL queries. Tasks including sentiment analysis, text classiication and translation. There is also open ended function that allows users to customize the prompt opening endless possiblities.
In this case we will use `ai_summarize` function to extract the key topics from all the reviews.

In [ ]:
%sql
select rating, combined_ratings, ai_summarize(array_join(combined_ratings, ', ')) review_summary
from collected_reviews

Using the query from above we will create a table in the gold layer.

In [ ]:
%sql
create or replace table gold.provider_review_summary
select rating, ai_summarize(array_join(combined_ratings, ', ')) review_summary
from collected_reviews

####Congratulations on Completing the Notebook